<a href="https://colab.research.google.com/github/Noors-lab/toll-plaza-vision-system/blob/main/building_interface_pipeline_for_toll_plaza" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# connecting to drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install ultralytics -q
!pip install ultralytics laptrack -q

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.5 MB/s eta 0:00:00


# Building the entire interface pipeline

In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque

# ─── CONFIG ───
MODEL_PATH = '/content/drive/MyDrive/toll_plaza_runs/v1/weights/best.pt'
VIDEO_PATH = '/content/drive/MyDrive/vision_data/day_cctv_0:00:41.mp4'
OUTPUT_PATH = '/content/drive/MyDrive/inference_results/phase3_test.mp4'

CLASSES = ['car', 'motorcycle', 'bus', 'truck', 'microbus']
COLORS = {
    'car':        (0, 255, 0),
    'motorcycle': (255, 165, 0),
    'bus':        (0, 0, 255),
    'truck':      (255, 0, 255),
    'microbus':   (0, 255, 255),
}

CONF_THRESHOLD = 0.4
SMOOTH_WINDOW = 5      # class smoothing over N frames
LINE_Y = 0.6           # virtual counting line at 60% of frame height

# ─── TRACKING STATE ───
track_history = defaultdict(lambda: deque(maxlen=SMOOTH_WINDOW))
counted_ids = set()
vehicle_counts = defaultdict(int)

# ─── LOAD MODEL ───
model = YOLO(MODEL_PATH)
print("Model loaded!")

# ─── VIDEO ───
cap = cv2.VideoCapture(VIDEO_PATH)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

line_y = int(h * LINE_Y)
frame_count = 0

print("Processing video...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # ─── DETECTION + TRACKING ───
    results = model.track(frame, persist=True, conf=CONF_THRESHOLD, verbose=False)

    if results[0].boxes is not None and results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.cpu().numpy().astype(int)
        class_ids = results[0].boxes.cls.cpu().numpy().astype(int)

        for box, track_id, class_id in zip(boxes, track_ids, class_ids):
            x1, y1, x2, y2 = map(int, box)
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            # ─── CLASS SMOOTHING ───
            track_history[track_id].append(class_id)
            smoothed_class = max(set(track_history[track_id]),
                                key=list(track_history[track_id]).count)
            class_name = CLASSES[smoothed_class]
            color = COLORS[class_name]

            # ─── LINE CROSSING ───
            if track_id not in counted_ids and cy > line_y:
                counted_ids.add(track_id)
                vehicle_counts[class_name] += 1

            # ─── DRAW BOX ───
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f'{class_name} #{track_id}',
                       (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX,
                       0.5, color, 2)

            # ─── DRAW CENTER DOT ───
            cv2.circle(frame, (cx, cy), 4, color, -1)

    # ─── DRAW COUNTING LINE ───
    cv2.line(frame, (0, line_y), (w, line_y), (0, 255, 255), 2)
    cv2.putText(frame, 'COUNTING LINE', (10, line_y - 10),
               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

    # ─── DRAW COUNTS ───
    y_offset = 30
    cv2.putText(frame, f'Frame: {frame_count}', (10, y_offset),
               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
    y_offset += 25
    for cls, count in vehicle_counts.items():
        cv2.putText(frame, f'{cls}: {count}', (10, y_offset),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLORS[cls], 2)
        y_offset += 25

    out.write(frame)

cap.release()
out.release()

print(f"\nDone! {frame_count} frames processed")
print("\nFinal vehicle counts:")
for cls, count in vehicle_counts.items():
    print(f"  {cls}: {count}")
print(f"\nOutput saved to: {OUTPUT_PATH}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model loaded!
Processing video...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 234ms
Prepared 1 package in 55ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


Done! 1001 frames processed

Final vehicle counts:
  car: 133
  motorcycle: 6
  truck: 4

Output saved to: /content/drive/MyDrive/inference_results/phase3_test.mp4
